In [1]:
# django shell 환경설정
import os
import django
os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings'
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'

django.setup()

# 페이징 처리
-  대량의 데이터를 여러 페이지로 나눠서 출력하는 것.
-  Django에서는 Paginator와 Page 클래스를 통해 처리한다.
  
## Paginator 클래스
- 전체 페이징 처리를 관리하는 클래스
- 전체 데이터관련 정보, 각 페이지당 보여줄 데이터의 정보 등을 제공

## Page 클래스
- 한페이지에대한 데이터를 관리
- Paginator를 통해서 생성.
    - `Pagenator객체.page(페이지 번호)`
- iterable 타입. 페이지에 속한 데이터들을 제공
- Page객체.object_list 속성: 페이지가 가진 데이터들을 List로 반환

In [2]:
from django.core.paginator import Paginator

In [3]:
data_list = list("0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")
print(len(data_list))
data_list[:5]

62


['0', '1', '2', '3', '4']

In [ ]:
# Paginator 객체 생성
pn = Paginator(data_list, 5) # 전체 데이터, 한 페이지에 보여줄 데이터 개수
# 전체 데이터(iterable): QuerySet - lazy loading(실제 값을 사용할 때 DB에서 조회해 옴)
# 한 번에 모든  데이터를 읽는 것이 아니라, 페이지에서 보여주기 위해 필요한 데이터만 그때그때 읽어옴

In [7]:
print("전체 데이터 개수:", pn.count)
print("총 페이지 수:", pn.num_pages)
print("시작 페이지 번호 ~ 끝 페이지 번호의 범위:", pn.page_range)

전체 데이터 개수: 62
총 페이지 수: 13
시작 페이지 번호 ~ 끝 페이지 번호의 범위: range(1, 14)


In [8]:
for p in pn.page_range:
    print(p, end=',')

1,2,3,4,5,6,7,8,9,10,11,12,13,

In [ ]:
# 특정 Page 객체를 생성
page1 = pn.page(1) # 조회 페이지 번호 -> Page 객체로 반환
print(type(page1))
page1

<class 'django.core.paginator.Page'>


<Page 1 of 13>

In [10]:
page10 = pn.page(10)
page10

<Page 10 of 13>

In [ ]:
page13 = pn.page(13)
page13 # 현재 페이지 of 전체 페이지 수

<Page 13 of 13>

In [13]:
# 없는 페이지 조회: error - EmptyPage  
try:
    pn.page(100)
except:
    print("100은 없는 페이지입니다.")                             

100은 없는 페이지입니다.


In [14]:
# Page가 가진 데이터들을 조회
# Page: Iterable 타입
for data in page1:
    print(data, end=', ')

0, 1, 2, 3, 4, 

In [15]:
for data in page10:
    print(data, end=', ')


J, K, L, M, N, 

In [16]:
for data in page13:
    print(data, end=', ')

Y, Z, 

In [ ]:
# index로 조회
# Page: Subscriptale
page1[0]
page1[3]
page1[1:3]

['1', '2']

In [ ]:
# Page의 데이터들을 리스트로 반환
page1.object_list
# = list(page1)

['0', '1', '2', '3', '4']

## 이전/다음 페이지가 있는지 여부
- `Page객체.has_previous()` / `Page객체.has_next()`
- 1페이지: 이전페이지? X, 다음페이지? O
- 중간 페이지: 이전페이지? O, 다음페이지? O
- 마지막 페이지: 이전? O, 다음페이지? X

In [24]:
# 시작 페이지(첫번째)
page1.has_previous(), page1.has_next()

(False, True)

In [25]:
# 중간 페이지
page10.has_previous(), page10.has_next()

(True, True)

In [26]:
# 마지막 페이지
page13.has_previous(), page13.has_next() 

(True, False)

## 이전/다음페이지 번호 조회
- **Page객체.number:** 현재 페이지 번호
- **Page객체.previous_page_number():** 이전페이지 번호 조회
- **page객체.next_page_number():** 다음페이지 번호 조회

In [27]:
# 현재 페이지 번호
print(page1.number, page10.number, page13.number)

1 10 13


In [28]:
# 이전페이지번호
print("이전 페이지 번호:", page10.previous_page_number())
print("현재 페이지 번호:", page10.number)
print("다음 페이지 번호:", page10.next_page_number())

이전 페이지 번호: 9
현재 페이지 번호: 10
다음 페이지 번호: 11


In [ ]:
# error - EmptyPage
# page1.previous_page_number # 이전 페이지가 없는 경우
# page13.next_page_number() # 다음 페이지가 없는 경우

In [29]:
if page1.has_previous():
    print("이전:",page1.previous_page_number())

if page1.has_next():
    print("다음:", page1.next_page_number())    

다음: 2


In [30]:
if page13.has_previous():
    print("이전:", page13.previous_page_number())

if page13.has_next():
    print("다음:", page13.next_page_number())

이전: 12


## 각 페이지별 데이터를 출력(조회)

In [ ]:
from django.core.paginator import Paginator

data_list = list("0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")

paginate_by = 3 # 한 페이지에 보여줄 데이터 수

pn = Paginator(data_list, paginate_by)

print(pn.page_range)
# 페이지들 반복
for page_no in pn.page_range: # range(시작, 끝+1) 페이지 번호 제공 range()
    page = pn.page(page_no)
    print(f"------{page_no} 페이지--------")
    # 페이지 안의 데이터들 반복
    for data in page:
        print(data, end=', ')
    print()


range(1, 22)
------1 페이지--------
0, 1, 2, 
------2 페이지--------
3, 4, 5, 
------3 페이지--------
6, 7, 8, 
------4 페이지--------
9, a, b, 
------5 페이지--------
c, d, e, 
------6 페이지--------
f, g, h, 
------7 페이지--------
i, j, k, 
------8 페이지--------
l, m, n, 
------9 페이지--------
o, p, q, 
------10 페이지--------
r, s, t, 
------11 페이지--------
u, v, w, 
------12 페이지--------
x, y, z, 
------13 페이지--------
A, B, C, 
------14 페이지--------
D, E, F, 
------15 페이지--------
G, H, I, 
------16 페이지--------
J, K, L, 
------17 페이지--------
M, N, O, 
------18 페이지--------
P, Q, R, 
------19 페이지--------
S, T, U, 
------20 페이지--------
V, W, X, 
------21 페이지--------
Y, Z, 


# 현재 페이지(요청페이지)가 속한 page 그룹의 (page_range)에서의 시작 index와 끝 index를 조회

In [32]:
from django.core.paginator import Paginator

data_list = list("0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")

pn = Paginator(data_list, 5)
r = pn.page_range
r

range(1, 14)

In [33]:
r[0:10]

range(1, 11)

In [ ]:
# page group당 3개 페이지씩 묶는 경우

print(list(r[0:3]))  # 현재 page: 1 or 2 or 3 일 경우 그 페이지가 속한 페이지들의 index 조회
print(list(r[3:6]))  # 현재 page: 4 or 5 or 6 일 경우 그 페이지가 속한 페이지들의 index 조회
print(list(r[6:9]))  # 현재 page: 7 or 8 or 9 일 경우 그 페이지가 속한 페이지들의 index 조회

In [34]:
current_page = 6
page_group_count = 10 # 페이지 그룹당 페이지 수

start_index = int((current_page - 1)/page_group_count) * page_group_count
end_index = start_index + page_group_count

r = pn.page_range
pg = r[start_index:end_index]
list(pg)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [35]:
r

range(1, 14)

## Question, Choice Dummy 데이터 추가

In [36]:
from polls.models import Question, Choice

In [37]:
# 기존 데이터를 삭제
q_list = Question.objects.all()
for q in q_list:
    q.delete()

In [ ]:
print(Question.objects.all().count()) # 조회한 데이터 개수 - QuerySet.count()
print(Choice.objects.all().count())

0
0


In [39]:
# Question 402개 추가.
for i in range(1, 403):
    post = Question(question_text=f"질문 - {i}")
    post.save()

In [40]:
cnt = Question.objects.all().count()
print(cnt)

402


In [42]:
# 첫번째 데이터의 primary key (id) 조회
start_id = Question.objects.all().order_by("pk")[0].pk
start_id

10

In [43]:
# 각 문제당 보기 4개씩추가.
import random

for i in range(start_id, cnt + start_id):

    # 개별 문제당 Choice 4개를 생성해서 save()
    for j in range(4):
        c = Choice(
            choice_text=f"{j}번 보기입니다.",
            votes=random.randint(0, 100),
            question=Question(pk=i),
        )
        c.save()

In [44]:
print(Question.objects.all().count())
print(Choice.objects.all().count())

402
1608
